In [1]:
import os
import io
import copy
import random
from tqdm.auto import tqdm 

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
# Classes
CAT_CLASS = 3
DOG_CLASS = 5

# Transform
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.247, 0.243, 0.261])
])

# Load CIFAR10
train_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=True,
                                      download=True, transform=transform)
test_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=False,
                                     download=True, transform=transform)


def filter_cat_vs_dog(dataset, cat_class=CAT_CLASS, dog_class=DOG_CLASS):
    indices = [i for i, label in enumerate(dataset.targets) if label in [cat_class, dog_class]]
    new_targets = [1 if dataset.targets[i] == cat_class else 0 for i in indices]
    subset = Subset(dataset, indices)
    subset.targets = new_targets
    return subset


class CatDogDataset(Dataset):
    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, _ = self.subset[idx]
        label = self.subset.targets[idx]
        return img, label


# Prepare loaders
train_loader = DataLoader(CatDogDataset(filter_cat_vs_dog(train_dataset_full)), batch_size=64, shuffle=True)
test_loader = DataLoader(CatDogDataset(filter_cat_vs_dog(test_dataset_full)), batch_size=64, shuffle=False)

print("Train/Test dataset ready.")


Files already downloaded and verified
Files already downloaded and verified
Train/Test dataset ready.


In [3]:
def load_model(path="model.pth"):
    print("\nLoading pretrained ResNet18 model...")
    model = resnet18(weights=ResNet18_Weights.DEFAULT)

    # Modify final layer for 2 classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 2)

    # Freeze all layers except layer3, layer4, fc
    for param in model.parameters():
        param.requires_grad = False
    for layer in [model.layer3, model.layer4, model.fc]:
        for param in layer.parameters():
            param.requires_grad = True

    # Load fine-tuned weights if available
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"Loaded fine-tuned weights from '{path}'")
    else:
        print(f"Warning: '{path}' not found. Model is not fine-tuned yet.")

    return model.to(device)


model = load_model()
model.eval()



Loading pretrained ResNet18 model...
Loaded fine-tuned weights from 'model.pth'


/tmp/ipykernel_8275/3570508965.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path, map_location=device))


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
class Encoder(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.features = nn.Sequential(
            trained_model.conv1,
            trained_model.bn1,
            trained_model.relu,
            trained_model.maxpool,
            trained_model.layer1,
            trained_model.layer2,
            trained_model.layer3
        )

    def forward(self, x):
        with torch.no_grad():
            return self.features(x)


encoder = Encoder(model).to(device)
encoder.eval()


Encoder(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True

In [5]:
class MirrorBasicBlock(nn.Module):
    """Reverse of a ResNet BasicBlock, preserving residual skip."""
    def __init__(self, block):
        super().__init__()
        # Mirror convs
        self.deconv2 = self.mirror_conv(block.conv2)
        self.bn2 = nn.BatchNorm2d(block.conv2.in_channels)
        self.deconv1 = self.mirror_conv(block.conv1)
        self.bn1 = nn.BatchNorm2d(block.conv1.in_channels)
        self.relu = nn.ReLU(inplace=True)

        # Mirror downsample (if present)
        if block.downsample is not None:
            down = block.downsample[0]
            self.upsample = nn.Sequential(
                self.mirror_conv(down),
                nn.BatchNorm2d(down.in_channels)
            )
        else:
            self.upsample = None

    def mirror_conv(self, conv):
        """Create a ConvTranspose2d layer that mirrors a given Conv2d layer."""
        return nn.ConvTranspose2d(
            in_channels=conv.out_channels,
            out_channels=conv.in_channels,
            kernel_size=conv.kernel_size,
            stride=conv.stride,
            padding=conv.padding,
            output_padding=1 if conv.stride[0] > 1 else 0,
            bias=False,
        )

    def forward(self, x):
        identity = x
        out = self.deconv2(x)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.deconv1(out)
        out = self.bn1(out)

        if self.upsample is not None:
            identity = self.upsample(identity)

        out += identity
        out = self.relu(out)
        return out


In [6]:
class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()

        base = models.resnet18(weights=None)

        def mirror_layer(layer):
            blocks = [MirrorBasicBlock(b) for b in reversed(layer)]
            return nn.Sequential(*blocks)

        self.up3 = mirror_layer(base.layer3)
        self.up2 = mirror_layer(base.layer2)
        self.up1 = mirror_layer(base.layer1)

        # Mirror of initial conv (7x7 stride=2)
        self.final = nn.ConvTranspose2d(
            64, 3, kernel_size=7, stride=2, padding=3, output_padding=1
        )

        # Extra step: bring from 16×16 -> 32×32
        self.upsample_final = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.up3(x)
        x = self.up2(x)
        x = self.up1(x)
        x = self.final(x)
        x = self.upsample_final(x)
        return x

decoder = Decoder().to(device)
model = model.to(device)

In [7]:
# Directory for saving decoders and summary CSV
save_dir = "hyperparam_MR_alpha_sigmoid_decoders"
os.makedirs(save_dir, exist_ok=True)

# DataFrame to store training stats
summary_df = pd.DataFrame(columns=["alpha", "raw_CE", "raw_MSE", "mean_CE", "mean_MSE", "norm_CE", "norm_MSE", "L_Total"])
csv_path = os.path.join(save_dir, "training_summary.csv")

# Losses and training settings
criterion_ce = nn.CrossEntropyLoss()
criterion_mse = nn.MSELoss()
LR = 1e-4
NUM_EPOCHS = 1000
PATIENCE = 20
momentum = 0.9
epsilon = 1e-10  # slightly larger epsilon for stability

alphas = [round(a*0.01, 2) for a in range(0, 101)]

# Freeze encoder & classifier
for p in encoder.parameters():
    p.requires_grad = False
for p in model.parameters():
    p.requires_grad = False

encoder.eval()
model.eval()

# Train decoders for each alpha
for alpha in alphas:
    save_path = os.path.join(save_dir, f"decoder{alpha:.2f}.pth")
    if os.path.exists(save_path):
        print(f"Skipping alpha {alpha:.2f}, decoder already exists at {save_path}")
        continue

    print(f"\nTraining decoder for alpha = {alpha:.2f}\n")
    decoder_curr = Decoder().to(device)
    decoder_curr.train()
    optimizer = optim.Adam(decoder_curr.parameters(), lr=LR)

    mean_ce, mean_mse = 1.0, 1.0
    best_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(NUM_EPOCHS):
        running_ce, running_mse, total = 0.0, 0.0, 0

        for images, labels in tqdm(train_loader, desc=f"Alpha {alpha:.2f} Epoch {epoch+1}", leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            latent = encoder(images)
            recon = decoder_curr(latent)
            recon = recon * 2 - 1
            logits = model(recon)

            # Raw losses
            L_CE_raw = criterion_ce(logits, labels)
            L_MSE_raw = criterion_mse(recon, images)

            # Normalized losses with epsilon to avoid huge spikes
            L_CE_norm = L_CE_raw / (mean_ce + epsilon)
            L_MSE_norm = L_MSE_raw / (mean_mse + epsilon)
            L_Total = L_CE_norm + alpha * L_MSE_norm

            # Backprop
            L_Total.backward()
            optimizer.step()

            batch_sz = images.size(0)
            running_ce += L_CE_raw.item() * batch_sz
            running_mse += L_MSE_raw.item() * batch_sz
            total += batch_sz

        # Compute normalized epoch losses
        epoch_ce = running_ce / total
        epoch_mse = running_mse / total
        epoch_ce_norm = epoch_ce / (mean_ce + epsilon)
        epoch_mse_norm = epoch_mse / (mean_mse + epsilon)
        epoch_total_norm = epoch_ce_norm + alpha * epoch_mse_norm

        # Update running means
        mean_ce = momentum * mean_ce + (1 - momentum) * epoch_ce
        mean_mse = momentum * mean_mse + (1 - momentum) * epoch_mse

        # Early stopping based on normalized loss
        if epoch_total_norm < best_loss - epsilon:
            best_loss = epoch_total_norm
            best_state = decoder_curr.state_dict()
            patience_counter = 0
            epochs_trained = epoch + 1
            best_epoch_ce = epoch_ce
            best_epoch_mse = epoch_mse
            best_mean_ce = mean_ce
            best_mean_mse = mean_mse
            best_ce_norm = epoch_ce_norm
            best_mse_norm = epoch_mse_norm
            best_total_norm = epoch_total_norm
        else:
            patience_counter += 1

        # Print normalized values
        print(f"Epoch {epoch+1:03d} | CE={epoch_ce_norm:.8f} | MSE={epoch_mse_norm:.8f} | "
              f"L_Total={epoch_total_norm:.8f} | Patience={patience_counter}")

        if patience_counter >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    # Save best decoder for this alpha
    torch.save(best_state, save_path)
    print(f"Saved best decoder for alpha={alpha:.2f} at {save_path}")

    # Record stats in DataFrame
    summary_df = pd.concat([
        summary_df,
        pd.DataFrame([{
            "alpha": alpha,
            "raw_CE": best_epoch_ce,
            "raw_MSE": best_epoch_mse,
            "mean_CE": best_mean_ce,
            "mean_MSE": best_mean_mse,
            "norm_CE": best_ce_norm,
            "norm_MSE": best_mse_norm,
            "L_Total": best_total_norm
        }])
    ], ignore_index=True)

# Save summary CSV
summary_df.to_csv(csv_path, index=False)
print(f"\nTraining summary saved at {csv_path}")



Training decoder for alpha = 0.00



Alpha 0.00 Epoch 1:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 001 | CE=0.37507379 | MSE=1.12041623 | L_Total=0.37507379 | Patience=0


Alpha 0.00 Epoch 2:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 002 | CE=0.06975513 | MSE=1.12451639 | L_Total=0.06975513 | Patience=0


Alpha 0.00 Epoch 3:   0%|          | 0/157 [00:00<?, ?it/s]

... (11,194 outputs left out)

Alpha 1.00 Epoch 36:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 036 | CE=0.19326645 | MSE=1.12292478 | L_Total=1.31619123 | Patience=19


Alpha 1.00 Epoch 37:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 037 | CE=0.00530243 | MSE=0.95462553 | L_Total=0.95992796 | Patience=20
Early stopping triggered at epoch 37
Saved best decoder for alpha=1.00 at hyperparam_MR_alpha_sigmoid_decoders/decoder1.00.pth

Training summary saved at hyperparam_MR_alpha_sigmoid_decoders/training_summary.csv
